### **Big data análisis utilizando `HDFS`, `Apache Hive` y `Hue`**

#### **Análisis de Olimpiadas**

![Olympic Logo](https://pyt-blogs.imgix.net/2016/08/20140204000881966198-original.jpg)

#### **`Paso 1` - Despliegue contenedores en Docker**

In [ ]:
C:\Users\Alfonso>cd Documents

C:\Users\Alfonso\Documents>cd ecosistema-hadoop

C:\Users\Alfonso\Documents\ecosistema-hadoop>docker compose up
[+] Running 8/5
 ✔ Container database                   Created                                                                    0.0s
 ✔ Container namenode                   Created                                                                    0.0s
 ✔ Container datanode                   Created                                                                    0.0s
 ✔ Container hue                        Created                                                                    0.0s
 ✔ Container hive-metastore-postgresql  Created                                                                    0.0s
 ✔ Container hive-metastore             Created                                                                    0.0s
 ✔ Container hive-server                Created                                                                    0.0s
Attaching to database, datanode, hive-metastore, hive-metastore-postgresql, hive-server, hue, namenode, zeppelin
namenode                   | Configuring core
database                   | 2024-05-26 02:54:46+00:00 [Note] [Entrypoint]: Entrypoint script for MySQL Server 5.7.44-1.el7 started.
namenode                   |  - Setting hadoop.proxyuser.hue.hosts=*
namenode                   |  - Setting fs.defaultFS=hdfs://namenode:8020
namenode                   |  - Setting hadoop.proxyuser.hue.groups=*
namenode                   |  - Setting hadoop.http.staticuser.user=root
namenode                   | Configuring hdfs
namenode                   |  - Setting dfs.namenode.name.dir=file:///hadoop/dfs/name
namenode                   |  - Setting dfs.permissions.enabled=false
namenode                   |  - Setting dfs.webhdfs.enabled=true
namenode                   | Configuring yarn
...
...

#### **`Paso 2` - Descarga de archivos de datos que vamos a procesar**

In [ ]:
# Utilizaremos el contenedor "namenode" para realizar la descarga de archivos y posterior mmovimiento hacia HDFS
C:\Users\Alfonso\Documents\ecosistema-hadoop>docker exec -it namenode bash
root@52d0320b5821:/#
root@52d0320b5821:/# pwd
/
root@52d0320b5821:/# mkdir -p /home/hadoop/archivos
root@52d0320b5821:/#
root@52d0320b5821:/# ls -l /home/hadoop     # <------ Comprobamos que se haya creado la ruta
total 4
drwxr-xr-x 2 root root 4096 May 26 03:05 archivos
root@52d0320b5821:/#
root@52d0320b5821:/#
root@52d0320b5821:/# hdfs dfs -ls /
Found 2 items
drwxrwxr-x   - root supergroup          0 2024-05-26 02:57 /tmp
drwxr-xr-x   - root supergroup          0 2024-05-26 02:56 /user
root@52d0320b5821:/#
root@52d0320b5821:/# hdfs dfs -ls /user
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-05-26 02:56 /user/hive
root@52d0320b5821:/#
root@52d0320b5821:/# hdfs dfs -mkdir -p /user/proyectos/proyecto1/data      # <----- Creamos un directorio en HDFS
root@52d0320b5821:/#
root@52d0320b5821:/# hdfs dfs -mkdir -p /user/proyectos/proyecto1/tablas    # <----- Creamos un directorio en HDFS
root@52d0320b5821:/#
root@52d0320b5821:/# hdfs dfs -ls /user/proyectos/proyecto1      # <----- Compruebo que se hayan creado dichos directorios en HDFS
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-05-26 03:20 /user/proyectos/proyecto1/data
drwxr-xr-x   - root supergroup          0 2024-05-26 03:20 /user/proyectos/proyecto1/tablas
root@52d0320b5821:/#

In [ ]:
# Entrar al directorio
root@52d0320b5821:/# cd /home/hadoop/archivos
root@52d0320b5821:/home/hadoop/archivos#
# Descargamos de la red los archivos que vamos a utilizar
root@52d0320b5821:/home/hadoop/archivos# curl -fLO https://raw.githubusercontent.com/hivesample/sample/main/athlete_events.csv
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6640k  100 6640k    0     0  1292k      0  0:00:05  0:00:05 --:--:-- 1846k
root@52d0320b5821:/home/hadoop/archivos#
root@52d0320b5821:/home/hadoop/archivos# curl -fLO https://raw.githubusercontent.com/hivesample/sample/main/noc_regions.csv
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3595  100  3595    0     0   7324      0 --:--:-- --:--:-- --:--:--  7321
root@52d0320b5821:/home/hadoop/archivos#
root@52d0320b5821:/home/hadoop/archivos# ls -l
total 6648
-rw-r--r-- 1 root root 6799918 May 26 03:25 athlete_events.csv
-rw-r--r-- 1 root root    3595 May 26 03:25 noc_regions.csv
root@52d0320b5821:/home/hadoop/archivos#
# Luego copiamos los archivos a HDFS
root@52d0320b5821:/home/hadoop/archivos# hdfs dfs -copyFromLocal athlete_events.csv /user/proyectos/proyecto1/data
root@52d0320b5821:/home/hadoop/archivos#
root@52d0320b5821:/home/hadoop/archivos# hdfs dfs -copyFromLocal noc_regions.csv /user/proyectos/proyecto1/data
root@52d0320b5821:/home/hadoop/archivos#
root@52d0320b5821:/home/hadoop/archivos# hdfs dfs -ls /user/proyectos/proyecto1/data
Found 2 items
-rw-r--r--   3 root supergroup    6799918 2024-05-26 03:27 /user/proyectos/proyecto1/data/athlete_events.csv
-rw-r--r--   3 root supergroup       3595 2024-05-26 03:27 /user/proyectos/proyecto1/data/noc_regions.csv
root@52d0320b5821:/home/hadoop/archivos#

#### **`Paso 3` - Creación de tablas y carga de datos en Hue**

In [ ]:
# Creamos una base de datos llamada "olimpiadas" para albergar las tablas
CREATE DATABASE IF NOT EXISTS olimpiadas
COMMENT 'almacena datos sobre las olimpiadas'
LOCATION '/user/proyectos/proyecto1/olimpiadas.db';

In [ ]:
# Comprobamos que la base de datos se haya creado correctamente
root@0ab30c74254e:/# hdfs dfs -ls /user/proyectos/proyecto1
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-05-28 15:45 /user/proyectos/proyecto1/data
drwxr-xr-x   - root supergroup          0 2024-05-28 15:59 /user/proyectos/proyecto1/olimpiadas.db
root@0ab30c74254e:/#

In [ ]:
# Creamos la tabla athlete_events_final
# Recordar que 'separatorChar' nos indica el separador 
# y 'quoteChar' nos indica el caracter a ignorar que se encuentre entre los separadores

USE olimpiadas;

CREATE TABLE IF NOT EXISTS athlete_events_final (
ID INT,
Name string, 
Sex string, 
Age INT, 
Height INT, 
Weight INT,
Team string, 
NOC string, 
Games string, 
Year INT, 
Season string, 
City string, 
Sport string, 
Event string, 
Medal string)
COMMENT 'athlete events table'
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES 
(
    "separatorChar" = ",",
    "quoteChar"     = "\""
);
# TBLPROPERTIES ("skip.header.line.count"="1");

In [ ]:
# Verificamos que la tabla "athlete_events_final" se haya creado en la base de datos "olimpiadas"
root@0ab30c74254e:/# hdfs dfs -ls /user/proyectos/proyecto1/olimpiadas.db
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-05-28 16:04 /user/proyectos/proyecto1/olimpiadas.db/athlete_events_final
root@0ab30c74254e:/#

In [ ]:
# También podriamos verificarlo de esta manera
hive> show databases;
OK
default
olimpiadas
Time taken: 0.015 seconds, Fetched: 2 row(s)
hive> SHOW TABLES IN olimpiadas;
OK
athlete_events_final                            # <----------
Time taken: 0.138 seconds, Fetched: 2 row(s)
hive>

In [ ]:
# Carga de datos
LOAD DATA INPATH '/user/proyectos/proyecto1/data/athlete_events.csv' INTO TABLE athlete_events_final;

In [ ]:
# Colocar el primer registro como encabezado
ALTER TABLE athlete_events_final SET TBLPROPERTIES ("skip.header.line.count"="1");

In [ ]:
# Lanzamos una consulta para verificar que los datos se hayan cargado correctamente
SELECT * FROM athlete_events_final;

<center><img src="https://i.postimg.cc/XYkg2WmH/p33.png"></center>

Recordemos que al cargarse un archivo a una tabla, este archivo será "eliminado" de su ruta original para posicionarse en la ruta de la tabla donde fue cargado. Vemos que el archivo "athlete_events_final" fue eliminado de us ruta original "/user/proyectos/proyecto1/data/" para ubicarse ahora en la ruta "/user/proyectos/proyecto1/olimpiadas.db/athlete_events_final". Esto sucede tanto para las tablas internas como externas.

In [ ]:
root@0ab30c74254e:/# hdfs dfs -ls /user/proyectos/proyecto1/olimpiadas.db
Found 1 items
drwxr-xr-x   - root supergroup          0 2024-05-28 16:06 /user/proyectos/proyecto1/olimpiadas.db/athlete_events_final
root@0ab30c74254e:/#
root@0ab30c74254e:/tmp# hdfs dfs -ls /user/proyectos/proyecto1/olimpiadas.db/athlete_events_final # <----- El archivo se encuentra en la ruta de la tabla en la que fue cargado
Found 1 items
-rwxr-xr-x   3 root supergroup    6799918 2024-05-28 15:45 /user/proyectos/proyecto1/olimpiadas.db/athlete_events_final/athlete_events.csv
root@0ab30c74254e:/tmp#
root@0ab30c74254e:/tmp# hdfs dfs -ls /user/proyectos/proyecto1/data   # <---- El archivo ya no se encuentra en esta ruta
Found 1 items
-rw-r--r--   3 root supergroup       3595 2024-05-28 15:45 /user/proyectos/proyecto1/data/noc_regions.csv
root@0ab30c74254e:/tmp#

In [ ]:
# Creamos la tabla noc_regions
CREATE TABLE IF NOT EXISTS noc_regions (
NOC string,
region string,
notes string)
COMMENT 'noc regions table'
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ',';
#TBLPROPERTIES("skip.header.line.COUNT"="1"); 


In [ ]:
# Verificamos que la tabla "noc_regions" se haya creado en la base de datos "olimpiadas"
root@0ab30c74254e:/# hdfs dfs -ls /user/proyectos/proyecto1/olimpiadas.db
Found 2 items
drwxr-xr-x   - root supergroup          0 2024-05-28 16:06 /user/proyectos/proyecto1/olimpiadas.db/athlete_events_final
drwxr-xr-x   - root supergroup          0 2024-05-28 16:18 /user/proyectos/proyecto1/olimpiadas.db/noc_regions
root@0ab30c74254e:/#

In [ ]:
# Carga de datos
LOAD DATA INPATH '/user/proyectos/proyecto1/data/noc_regions.csv' INTO TABLE noc_regions;

In [ ]:
# Colocar el primer registro como encabezado
ALTER TABLE noc_regions SET TBLPROPERTIES ("skip.header.line.count"="1");

In [ ]:
# Lanzamos una consulta para verificar que los datos se hayan cargado correctamente
SELECT * FROM noc_regions;

<center><img src="https://i.postimg.cc/mrZyJqGb/p34.png"></center>

#### **`Paso 4` - Análisis de datos**

##### Distribución de edad para los deportistas con medalla de oro

In [ ]:
SELECT COUNT(Medal) AS Medals, Age 
FROM olimpiadas.athlete_events_final 
WHERE Medal = 'Gold' 
GROUP BY Age 
ORDER BY Age ASC;

<center><img src="https://i.postimg.cc/BbmHqVpT/p35.png"></center>

##### Medallas de oro para deportistas de más de 50 años según deportes

In [ ]:
SELECT Sport, Age 
FROM olimpiadas.athlete_events_final 
WHERE Medal = 'Gold' 
AND Age >= 50; 

<center><img src="https://i.postimg.cc/15Rfm04z/p36.png"></center>

##### Medallas femeninas por edición (temporada de verano) de los juegos

In [ ]:
SELECT COUNT(Medal) AS Medals, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'F' 
AND Season = 'Summer' 
AND Medal IN ('Bronze','Gold','Silver') 
GROUP BY Year ORDER BY Year ASC;

<center><img src="https://i.postimg.cc/SxJR8y6F/p37.png"></center>p37

##### 5 países con más medallas de oro

In [ ]:
SELECT COUNT(Medal) AS Medals, region 
FROM olimpiadas.athlete_events_final A 
JOIN olimpiadas.noc_regions N 
ON A.NOC = N.NOC  
WHERE Medal = 'Gold' 
GROUP BY region 
ORDER BY Medals DESC LIMIT 5;

<center><img src="https://i.postimg.cc/yYfWkJSy/p38.png"></center>

##### Disciplinas con mayor número de medallas de oro para USA

In [ ]:
SELECT COUNT(Medal) AS Medals, Event 
FROM olimpiadas.athlete_events_final A 
JOIN olimpiadas.noc_regions N 
ON A.NOC = N.NOC  
WHERE Medal = 'Gold' 
AND A.NOC = 'USA' 
GROUP BY Event 
ORDER BY Medals DESC LIMIT 5; 

<center><img src="https://i.postimg.cc/9fBw6NWc/p39.png"></center>

##### Variación de la participación masculina a lo largo del tiempo (Juegos Olimpicos de Verano)

In [ ]:
SELECT COUNT(Sex) AS Males, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'M' 
AND Season = 'Summer'
GROUP BY Year
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/j5dntQ5P/p40.png"></center>

##### Variación de la participación femenina a lo largo del tiempo (Juegos Olimpicos de Verano)

In [ ]:
SELECT COUNT(Sex) AS Females, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'F' 
AND Season = 'Summer'
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/dQfZcCKZ/p41.png"></center>

##### Promedio de edad de los deportistas masculinos a lo largo del tiempo

In [ ]:
SELECT AVG(Age) AS edad_promedio, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'M'
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/Gt64FWjx/p42.png"></center>

##### Promedio de edad de las deportistas femeninas a lo largo del tiempo

In [ ]:
SELECT AVG(Age) AS edad_promedio, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'F' 
GROUP BY Year 
ORDER BY Year ASC;

<center><img src="https://i.postimg.cc/Qt49t99P/p43.png"></center>

##### Promedio del peso de los deportistas masculinos a lo largo del tiempo

In [ ]:
SELECT AVG(Weight) AS promedio_peso, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'M' 
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/prm9nWKd/p44.png"></center>

##### Promedio del peso de las deportistas femeninas a lo largo del tiempo

In [ ]:
SELECT AVG(Weight) AS peso_promedio, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'F' 
AND Year > 1925 
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/ncQjFLHR/p45.png"></center>

##### Promedio de altura de los deportistas masculinos a lo largo del tiempo

In [ ]:
SELECT AVG(Height) AS promedio_altura, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'M' 
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/0yR6bqj8/p46.png"></center>

##### Promedio de altura de las deportistas femeninas a lo largo del tiempo

In [ ]:
SELECT AVG(Height) AS promedio_altura, Year 
FROM olimpiadas.athlete_events_final 
WHERE Sex = 'F' 
GROUP BY Year 
ORDER BY Year ASC; 

<center><img src="https://i.postimg.cc/pXT66gJJ/p47.png"></center>

##### Medallas de oro según pais

In [ ]:
SELECT COUNT(Medal) AS Medals, N.region AS pais 
FROM olimpiadas.athlete_events_final A 
JOIN olimpiadas.noc_regions N 
ON A.NOC = N.NOC  
WHERE Medal = 'Gold' 
GROUP BY N.region
HAVING COUNT(Medal) > 50
ORDER BY N.region;

<center><img src="https://i.postimg.cc/4NSSkB8k/p48.png"></center>

##### Medallas de plata según pais

In [ ]:
SELECT COUNT(Medal) AS Medals, N.region AS pais 
FROM olimpiadas.athlete_events_final A 
JOIN olimpiadas.noc_regions N 
ON A.NOC = N.NOC  
WHERE Medal = 'Silver' 
GROUP BY N.region
HAVING COUNT(Medal) > 50
ORDER BY N.region;

<center><img src="https://i.postimg.cc/L8g0HL7q/p49.png"></center>p49

##### Medallas de bronce según pais

In [ ]:
SELECT COUNT(Medal) AS Medals, N.region AS pais 
FROM olimpiadas.athlete_events_final A 
JOIN olimpiadas.noc_regions N 
ON A.NOC = N.NOC  
WHERE Medal = 'Bronze' 
GROUP BY N.region
HAVING COUNT(Medal) > 50
ORDER BY N.region;

<center><img src="https://i.postimg.cc/d0MNDDP5/p50.png"></center>